# Create ROIs for template matching

This notebook is used to define and save the ROIs used in the template-matching workflow. The ROIs are saved to `../results/roi_by_tilt.json` and later loaded in the template-matching notebook.

This notebook should only be rerun when the ROI positions need to be changed.

In [ ]:

%matplotlib widget

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import hyperspy.api as hs

import sys
import importlib

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.roi_utils as roi_utils

importlib.reload(roi_utils)

from src.roi_utils import (
    load_rois,
    save_rois_by_tilt,
)

In [ ]:
results_path = Path("../results")
results_path.mkdir(parents=True, exist_ok=True)

roi_path = results_path / "roi_by_tilt.json"

roi_colors = {
    "A": "red",
    "B": "aqua",
    "C": "mediumorchid",
    "D": "hotpink",
}

In [ ]:
# load data from 7 tilts
datasets = [
    {"tilt": +10, "path": Path("../../data/20260417_Ingrid/093932/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_18p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": +5,  "path": Path("../../data/20260417_Ingrid/100104/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_13p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": 0,   "path": Path("../../data/20260417_Ingrid/091336/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_8p2_Ty3p8_big_centered_masked.hspy")},
    {"tilt": -5,  "path": Path("../../data/20260417_Ingrid/101949/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_3p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": -10, "path": Path("../../data/20260417_Ingrid/103611/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m2p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": -15, "path": Path("../../data/20260417_Ingrid/105201/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m7p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": -20, "path": Path("../../data/20260417_Ingrid/110843/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m12p2_Ty3p8_cropped_centered_masked.hspy")},
]

In [ ]:
# %%
lazy = True
signals = {}

for dataset in datasets:

    tilt = dataset["tilt"]
    path = dataset["path"]

    print(f"Loading tilt {tilt}")

    signal = hs.load(
        str(path),
        lazy=lazy,
    )

    if lazy:
        signal.rechunk(
            nav_chunks=(32, 32),
            sig_chunks=(32, 32),
        )

    print(f"  shape={signal.data.shape}, dtype={signal.data.dtype}")

    signals[tilt] = signal

In [ ]:


roi_path = results_path / "roi_by_tilt.json"

if roi_path.exists():
        
        print(f"Loading ROIs")
        roi_by_tilt = load_rois(roi_path)

else:
    
    print("No ROI file. Define interactively")

    roi_by_tilt = {}

    for tilt, signal in signals.items():

        print(f"\n Tilt {tilt}")
        

        roi_A = hs.roi.RectangularROI(left=100.58, right=177.99, top=603.77, bottom=635.18)
        roi_B = hs.roi.RectangularROI(left=247.79, right=279.20, top=572.36, bottom=603.77)
        roi_C = hs.roi.RectangularROI(left=474.64, right=506.05, top=607.26, bottom=638.67)
        roi_D = hs.roi.RectangularROI(left=628.20, right=659.61, top=602.29, bottom=771.70)

        signal.plot(norm="symlog")

        roi_A.add_widget(signal, color="red")
        roi_B.add_widget(signal, color="aqua")
        roi_C.add_widget(signal, color="mediumorchid")
        roi_D.add_widget(signal, color="hotpink")

        roi_by_tilt[tilt] = {
            "A": roi_A,
            "B": roi_B,
            "C": roi_C,
            "D": roi_D
        }

In [ ]:
for tilt, signal in signals.items():

    print(f"Tilt {tilt}")

    # plot dataset
    signal.plot(norm="symlog")

    rois = roi_by_tilt[tilt]

    # add overlays
    for key, roi in rois.items():
        roi.add_widget(signal, color=roi_colors[key])

In [ ]:
# %%
save_rois_by_tilt(
    roi_by_tilt,
    roi_path,
)

print(f"Saved ROIs to: {roi_path}")